# Environment Setup & Railway Data Generator

In [1]:
import os
from pathlib import Path
import boto3
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from datetime import datetime, timedelta
import random

# Load credentials from the root project folder
load_dotenv(Path("..") / ".env", override=True)

# Required security guardrails
os.environ["AWS_REQUEST_CHECKSUM_CALCULATION"] = "when_required"
os.environ["AWS_RESPONSE_CHECKSUM_VALIDATION"] = "when_required"

AWS_KEY = os.getenv("AWS_ACCESS_KEY_ID", "")
AWS_SECRET = os.getenv("AWS_SECRET_ACCESS_KEY", "")
S3_ENDPOINT = os.getenv("S3_ENDPOINT", "")
BUCKET = os.getenv("S3_BUCKET_NAME", "")

# --- CONFIGURATION ---
LOCAL_FILE = "railway_telemetry.csv"
S3_KEY = "railway/railway_telemetry.csv"

# Initialize S3 client
s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_KEY,
    aws_secret_access_key=AWS_SECRET,
    endpoint_url=S3_ENDPOINT,
    verify=False
)


def generate_synthetic_rail_data(output_path, num_samples=2000):
    print("🚆 Generating railway telemetry dataset...")

    np.random.seed(42)
    random.seed(42)

    # -----------------------------
    # Generate sensor data
    # -----------------------------

    num_normal = int(num_samples * 0.75)
    num_vuln = int(num_samples * 0.15)
    num_emergency = num_samples - num_normal - num_vuln

    # Normal tracks
    df_normal = pd.DataFrame({
        "vibration_hz": np.random.normal(50, 5, num_normal),
        "acoustic_emission_db": np.random.normal(40, 3, num_normal),
        "rail_strain_mu": np.random.normal(120, 10, num_normal),
        "track_temperature_c": np.random.normal(32, 4, num_normal),
        "alignment_deviation_mm": np.random.normal(0, 0.5, num_normal),
    })

    # High wear
    df_vuln = pd.DataFrame({
        "vibration_hz": np.random.normal(68, 4, num_vuln),
        "acoustic_emission_db": np.random.normal(52, 4, num_vuln),
        "rail_strain_mu": np.random.normal(142, 8, num_vuln),
        "track_temperature_c": np.random.normal(48, 3, num_vuln),
        "alignment_deviation_mm": np.random.normal(2.1, 0.8, num_vuln),
    })

    # Emergency
    df_emergency = pd.DataFrame({
        "vibration_hz": np.random.normal(85, 8, num_emergency),
        "acoustic_emission_db": np.random.normal(65, 6, num_emergency),
        "rail_strain_mu": np.random.normal(175, 15, num_emergency),
        "track_temperature_c": np.random.normal(62, 5, num_emergency),
        "alignment_deviation_mm": np.random.choice(
            [
                np.random.normal(-4.5, 0.5),
                np.random.normal(4.5, 0.5),
            ],
            num_emergency,
        ),
    })

    df = pd.concat([df_normal, df_vuln, df_emergency], ignore_index=True)
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)

    # -------------------------------------------------------
    # Generate failure probability instead of 0/1 prediction
    # -------------------------------------------------------

    score = (
        0.35 * (df["vibration_hz"] / 90)
        + 0.30 * (df["rail_strain_mu"] / 180)
        + 0.20 * (abs(df["alignment_deviation_mm"]) / 5)
        + 0.15 * (df["track_temperature_c"] / 70)
    )

    score = np.clip(score, 0, 1)

    noise = np.random.normal(0, 0.04, len(df))

    df["failure_probability"] = np.clip(score + noise, 0, 1).round(2)

    # -----------------------------
    # Risk Level
    # -----------------------------

    def risk(prob):
        if prob < 0.30:
            return "LOW"
        elif prob < 0.60:
            return "MEDIUM"
        elif prob < 0.85:
            return "HIGH"
        else:
            return "CRITICAL"

    df["risk_level"] = df["failure_probability"].apply(risk)

    # -----------------------------
    # Track metadata
    # -----------------------------

    track_ids = [f"TRK-{i:03d}" for i in range(1, 51)]

    df["track_id"] = np.random.choice(track_ids, len(df))

    df["section"] = np.random.choice(
        ["North", "South", "East", "West"],
        len(df),
    )

    # -----------------------------
    # Timestamp
    # -----------------------------

    start = datetime.now() - timedelta(days=2)

    df["timestamp"] = [
        start + timedelta(seconds=i * 60)
        for i in range(len(df))
    ]

    # -----------------------------
    # Fault Type
    # -----------------------------

    def fault(row):

        if row["risk_level"] == "LOW":
            return "Normal"

        if abs(row["alignment_deviation_mm"]) > 4:
            return "Track Misalignment"

        if row["vibration_hz"] > 80:
            return "Rail Crack"

        if row["rail_strain_mu"] > 160:
            return "Structural Stress"

        return "Excessive Wear"

    df["fault_type"] = df.apply(fault, axis=1)

    # -----------------------------
    # Inspection Status
    # -----------------------------

    status = []

    for r in df["risk_level"]:

        if r == "LOW":
            status.append("Completed")

        elif r == "MEDIUM":
            status.append(random.choice([
                "Completed",
                "Pending"
            ]))

        elif r == "HIGH":
            status.append(random.choice([
                "Pending",
                "In Progress"
            ]))

        else:
            status.append(random.choice([
                "Pending",
                "In Progress"
            ]))

    df["inspection_status"] = status

    # -----------------------------
    # Recommended Action
    # -----------------------------

    def action(r):

        if r == "LOW":
            return "Monitor"

        elif r == "MEDIUM":
            return "Schedule Inspection"

        elif r == "HIGH":
            return "Repair Soon"

        else:
            return "Immediate Shutdown"

    df["recommended_action"] = df["risk_level"].apply(action)

    # -----------------------------
    # Assigned Team
    # -----------------------------

    df["assigned_team"] = np.random.choice(
        [
            "Team Alpha",
            "Team Bravo",
            "Team Charlie",
            "Unassigned",
        ],
        len(df),
        p=[0.3, 0.3, 0.2, 0.2],
    )

    # -----------------------------
    # Maintenance
    # -----------------------------

    df["last_maintenance_days"] = np.random.randint(
        1,
        180,
        len(df),
    )

    df["action_taken"] = np.where(
        df["inspection_status"] == "Completed",
        "Yes",
        "No",
    )

    # -----------------------------
    # Reorder columns
    # -----------------------------

    df = df[
        [
            "timestamp",
            "track_id",
            "section",
            "vibration_hz",
            "acoustic_emission_db",
            "rail_strain_mu",
            "track_temperature_c",
            "alignment_deviation_mm",
            "failure_probability",
            "risk_level",
            "fault_type",
            "inspection_status",
            "recommended_action",
            "assigned_team",
            "last_maintenance_days",
            "action_taken",
        ]
    ]

    df.to_csv(output_path, index=False)

    print(f"✅ Generated {len(df)} records")
    print(f"📁 Saved to {output_path}")


if os.path.exists(LOCAL_FILE):
    os.remove(LOCAL_FILE)

generate_synthetic_rail_data(LOCAL_FILE)

🚆 Generating railway telemetry dataset...
✅ Generated 2000 records
📁 Saved to railway_telemetry.csv


# Upload Dataset to Vayu Object Storage

In [2]:
import os
from pathlib import Path
import boto3
import botocore.config
from dotenv import load_dotenv

# Force reload the .env variables
load_dotenv(Path("..") / ".env", override=True)

AWS_KEY = os.getenv("AWS_ACCESS_KEY_ID", "")
AWS_SECRET = os.getenv("AWS_SECRET_ACCESS_KEY", "")
S3_ENDPOINT = os.getenv("S3_ENDPOINT", "")
BUCKET = os.getenv("S3_BUCKET_NAME", "")
S3_KEY = os.getenv("S3_DATASET_KEY", "railway/railway_telemetry.csv")

LOCAL_FILE = "railway_telemetry.csv"

# Build configuration using the absolute, fully-qualified module path
s3_config = botocore.config.Config(
    signature_version='s3v4',
    s3={'addressing_style': 'path'}
)

# Initialize the client cleanly
s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_KEY,
    aws_secret_access_key=AWS_SECRET,
    endpoint_url=S3_ENDPOINT,
    config=s3_config
)
print("🚀 S3 Client successfully initialized without errors!")

🚀 S3 Client successfully initialized without errors!


In [3]:
import os
from pathlib import Path

import boto3
from botocore.config import Config
from botocore.exceptions import ClientError
from dotenv import load_dotenv

# =====================================================
# Load .env
# =====================================================

env_path = Path("../.env")

print(f"Loading env from: {env_path.resolve()}")

if not env_path.exists():
    raise FileNotFoundError(f".env not found at {env_path.resolve()}")

load_dotenv(env_path, override=True)

# =====================================================
# Read credentials
# =====================================================

AWS_KEY = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET = os.getenv("AWS_SECRET_ACCESS_KEY")
S3_ENDPOINT = os.getenv("S3_ENDPOINT")
BUCKET = os.getenv("S3_BUCKET_NAME")
S3_KEY = os.getenv("S3_DATASET_KEY", "railway/railway_telemetry.csv")

LOCAL_FILE = "railway_telemetry.csv"

# =====================================================
# Validate
# =====================================================

print("\n========== Configuration ==========")
print("AWS Key       :", AWS_KEY[:6] + "******" if AWS_KEY else None)
print("AWS Secret    :", "Loaded" if AWS_SECRET else None)
print("Endpoint      :", S3_ENDPOINT)
print("Bucket        :", BUCKET)
print("Object Key    :", S3_KEY)
print("Local File    :", LOCAL_FILE)
print("===================================\n")

required = {
    "AWS_ACCESS_KEY_ID": AWS_KEY,
    "AWS_SECRET_ACCESS_KEY": AWS_SECRET,
    "S3_ENDPOINT": S3_ENDPOINT,
    "S3_BUCKET_NAME": BUCKET,
}

missing = [k for k, v in required.items() if not v]

if missing:
    raise ValueError(f"Missing environment variables: {missing}")

# =====================================================
# Create S3 client
# =====================================================

config = Config(
    signature_version="s3v4"
)

s3 = boto3.client(
    "s3",
    endpoint_url=S3_ENDPOINT,
    aws_access_key_id=AWS_KEY,
    aws_secret_access_key=AWS_SECRET,
    config=config,
)

# =====================================================
# Check local file
# =====================================================

if not os.path.exists(LOCAL_FILE):
    raise FileNotFoundError(f"{LOCAL_FILE} not found")

print("✓ Local file found")

# =====================================================
# Test S3 connection
# =====================================================

print("\nTesting S3 connection...")

try:
    response = s3.list_objects_v2(
        Bucket=BUCKET,
        MaxKeys=1
    )

    print("✓ Connected to bucket successfully")

except ClientError as e:
    print("\n❌ Could not access bucket")
    print(e.response["Error"])
    raise

# =====================================================
# Upload
# =====================================================

print("\nUploading...")

try:
    s3.upload_file(
        Filename=LOCAL_FILE,
        Bucket=BUCKET,
        Key=S3_KEY
    )

    print("\n🎉 Upload Successful!")
    print(f"s3://{BUCKET}/{S3_KEY}")

except ClientError as e:
    print("\n❌ Upload failed")
    print(e.response["Error"])

except Exception as e:
    print("\n❌ Unexpected error")
    print(e)

Loading env from: /home/jovyan/move-it/.env

========== Configuration ==========
AWS Key       : AKIA37******
AWS Secret    : Loaded
Endpoint      : https://team18.ipstorage.tatacommunications.com
Bucket        : xoxeb-bucket
Object Key    : railway/railway_telemetry.csv
Local File    : railway_telemetry.csv

✓ Local file found

Testing S3 connection...
✓ Connected to bucket successfully

Uploading...

🎉 Upload Successful!
s3://xoxeb-bucket/railway/railway_telemetry.csv


# Download Dataset from Vayu Object Storage

In [4]:
def download_railway_dataset(bucket, s3_key, local_file):
    print(f"🛰️ Downloading s3://{bucket}/{s3_key} → {local_file}")
    try:
        s3.download_file(bucket, s3_key, local_file)
        print("🎉 DONE: Track dataset downloaded locally and ready for analysis scripts.")
        
        # Quick validation check to review structure shape mapping
        df_preview = pd.read_csv(local_file)
        print(f"\n📋 Preview of ingested telemetry array features:")
        print(df_preview.head(3))
    except Exception as e:
        print(f"❌ Error pulling object array payload from storage bucket domain: {e}")

# Run download task routine to verify bucket connectivity works transparently
download_railway_dataset(BUCKET, S3_KEY, "downloaded_railway_telemetry.csv")

🛰️ Downloading s3://xoxeb-bucket/railway/railway_telemetry.csv → downloaded_railway_telemetry.csv
🎉 DONE: Track dataset downloaded locally and ready for analysis scripts.

📋 Preview of ingested telemetry array features:
                    timestamp track_id section  vibration_hz  \
0  2026-07-10 20:08:14.880864  TRK-049    East     84.183176   
1  2026-07-10 20:09:14.880864  TRK-033   South     49.200307   
2  2026-07-10 20:10:14.880864  TRK-033   South     57.453631   

   acoustic_emission_db  rail_strain_mu  track_temperature_c  \
0             53.446596      180.718670            61.740226   
1             42.757463      120.771565            32.595507   
2             39.178021      121.994240            35.282214   

   alignment_deviation_mm  failure_probability risk_level          fault_type  \
0                4.751616                 0.90   CRITICAL  Track Misalignment   
1                0.367238                 0.50     MEDIUM      Excessive Wear   
2                0.6977